In [54]:
import bw2data as bd
import bw2io as bi
from bw2io import BW2Package
from bw2calc import LCA
from pathlib import Path
import bw2analyzer as ba
import pandas as pd
from IPython.display import display
import re
import hashlib
import pickle
import numpy as np
import ipywidgets as widgets
import plotly.graph_objects as go
from plotly.subplots import make_subplots


In [55]:
PROJECT_NAME = "Lasers4MaaS"
ECOINVENT_DATABASE_NAME = "ecoinvent-3.12-cutoff"

bd.projects.set_current(PROJECT_NAME)
if ECOINVENT_DATABASE_NAME not in bd.databases:
    bi.import_ecoinvent_release(version="3.12", system_model="cutoff")
else:
    print(f"Database {ECOINVENT_DATABASE_NAME} already exists, skipping import.")

Database ecoinvent-3.12-cutoff already exists, skipping import.


In [56]:
ei = bd.Database(ECOINVENT_DATABASE_NAME)

### Variables: bio


In [57]:
bio = bd.Database("ecoinvent-3.12-biosphere")

## L4M R&D: parameter sets per use case
**Goal:** identify a set of optimised welding parameters for each use case (Fusion welding, Pharma welding, Battery cooling welding).
**Reference flow:** 1 set of optimised parameters for one use case.
**Scenarios per use case:**
- *SoA R&D*: all experiments run physically in the lab with the standard laser.
- *L4M-enabled R&D*: fewer physical experiments, plus digital lab runs, with the joining done by Dynamic Beam Shaping (DBS).

Data come from `databases/L4M_database.bw2package` (`L4M system`).

In [58]:
PACKAGE_DIR = Path("databases")
L4M_NAME = "L4M system"
METALLURGIC_NAME = "Mettalurgic test (CIVAN)"      # spelling as stored in the packages
CT_SCAN_NAME = "CT-scan test"
FOREGROUND_DBS = [L4M_NAME, METALLURGIC_NAME, CT_SCAN_NAME]

# Set to True to delete and re-import L4M system from the package (discards edits made in the project)
REIMPORT_L4M = False

def ensure_package(db_name, package_file, force=False):
    """Import a .bw2package unless the database already holds activities."""
    if db_name in bd.databases and len(bd.Database(db_name)) > 0 and not force:
        print(f"{db_name}: {len(bd.Database(db_name))} activities already in the project, skipping import.")
        return
    if db_name in bd.databases:
        del bd.databases[db_name]
    BW2Package.import_file(str(PACKAGE_DIR / package_file))
    print(f"{db_name}: imported {len(bd.Database(db_name))} activities from {package_file}")

# L4M system links to the other two databases, so they come first
ensure_package(CT_SCAN_NAME, "CT-scan test.bw2package")
ensure_package(METALLURGIC_NAME, "Metallurgic test (CIVAN).bw2package")
ensure_package(L4M_NAME, "L4M_database.bw2package", force=REIMPORT_L4M)

for db in FOREGROUND_DBS:
    for a in bd.Database(db):
        for e in a.exchanges():          # raises UnknownObject for a dangling link
            e.input
    bd.Database(db).process()            # rebuild the matrices so freshly imported dependencies are picked up
l4m = bd.Database(L4M_NAME)

CT-scan test: 5 activities already in the project, skipping import.
Mettalurgic test (CIVAN): 9 activities already in the project, skipping import.
L4M system: 11 activities already in the project, skipping import.


### Impact indicators (same EF 3.1 selection as `joining_task.ipynb`)

In [59]:
most_used_indicators = [
    'climate change: total',
    'particulate matter formation',
    'acidification',
    'eutrophication: terrestrial',
    'photochemical oxidant formation: human health',
    'human toxicity: carcinogenic'
]
ef31_methods = [
    m for m in bd.methods
    if ('EF v3.1' in str(m) or 'EF 3.1' in str(m))
    and 'no LT' not in str(m)
    and any(indicator in str(m) for indicator in most_used_indicators)
]
method_units = {m[2]: bd.Method(m).metadata.get("unit", "") for m in ef31_methods}
indicators = [m[2] for m in ef31_methods]
DEFAULT_INDICATOR = next((i for i in indicators if i.startswith("climate change")), indicators[0])
short = lambda s: s.split(" (EF")[0]          # drop the "(EF v3.1 - IPCC 2021)" suffix in titles

### Saved processes and their reference flows
Each saved process is loaded as it is stored in its database. The comment on each line gives the reference flow: the amount of its reference product that the process produces, and the unit. All input amounts of a process refer to this reference flow.
The next cell builds the R&D scenario systems from these processes. The Sankey and regional cells below stay in the notebook as raw cells (disabled).

In [60]:
def load(db, code):
    return bd.Database(db).get(code)

# Reference flow = production exchange of the process: amount and unit of its reference product
l4m_processes = {
    "001_Physical lab (battery cooling pre-processing + welding)": load(L4M_NAME, "a7cf0ea95c2a499aa8cf9eb0ba72142c"),  # 1 unit of "1 set of optimised parameters"
    "001_Physical_Cutting":       load(L4M_NAME, "97a5ed8baec4406f879d5cff60e87345_copy1"),  # 500 unit of "Cut coupon"
    "001_Physical_Laser welding": load(L4M_NAME, "5a81aade45924c56854be233364a0435_copy1"),  # 250 unit of "Weld coupon"
    "001_Pyshical_Cleaning":      load(L4M_NAME, "fbbf6041c24c4e28b6c23a3ab0eebf0a_copy1"),  # 500 unit of "Clean coupon"
    "Battery cooling welding":    load(L4M_NAME, "68f4a5e21189435ba9b0a376e068edbc"),        # 1 unit of "Battery cooling plate"
    "CT-scan X-ray":              load(L4M_NAME, "93d9467878cb43ba9798a256e98512f5_copy1"),  # 1 unit of "Tested coupon" (location GB)
    "Digital lab processes":      load(L4M_NAME, "d273f34fee714ae0b25dbfb39576ab5b"),        # 1 unit of "Digital lab processes"
    "Fusion welding":             load(L4M_NAME, "68f4a5e21189435ba9b0a376e068edbc_copy2"),  # 1 unit of "Pipe welding"
    "Pharma welding":             load(L4M_NAME, "68f4a5e21189435ba9b0a376e068edbc_copy1"),  # 1 unit of "Freeze dryer shelf"
    "Platform":                   load(L4M_NAME, "50a84844a3de4677926d511e171e31d5"),        # 1 unit of "Platform"
    "Prediction":                 load(L4M_NAME, "8fbb7937f5224de9b836ae54858b6fe1"),        # 1 unit of "Prediction"
}

metallurgic_processes = {
    "01_Cutting":      load(METALLURGIC_NAME, "97a5ed8baec4406f879d5cff60e87345"),  # 132 unit of "Cut coupon"
    "02_Cleaning":     load(METALLURGIC_NAME, "fbbf6041c24c4e28b6c23a3ab0eebf0a"),  # 1 unit of "Clean coupon"
    "03_Laser welding": load(METALLURGIC_NAME, "5a81aade45924c56854be233364a0435"), # 1 unit of "Weld coupon"
    "04_Cutting":      load(METALLURGIC_NAME, "7faf722864b94ed797cdd406d22120db"),  # 3 unit of "Weld cross-sections"
    "05_Hot-mounting": load(METALLURGIC_NAME, "06923c5d5a4d416d87c1702173dc648a"),  # 1 unit of "Mounted sample"
    "06_Grinding":     load(METALLURGIC_NAME, "f00138e49c5843caa7a1e7d8f33f5986"),  # 6 unit of "Grinded sample"
    "07_Polishing":    load(METALLURGIC_NAME, "53c6381979c942e79ed1f043a8b184ed"),  # 6 unit of "Polished sample"
    "08_Etching":      load(METALLURGIC_NAME, "92039648c9a64364ab22f228c0f4b44d"),  # 18 unit of "Etched sample"
    "09_Test":         load(METALLURGIC_NAME, "32bb94122a4542ad8af12fb8ea34a26e"),  # 18 unit of "Tested sample"
}

ct_scan_processes = {
    "01_Cutting":       load(CT_SCAN_NAME, "97a5ed8baec4406f879d5cff60e87345"),  # 132 unit of "Cut coupon"
    "02_Cleaning":      load(CT_SCAN_NAME, "fbbf6041c24c4e28b6c23a3ab0eebf0a"),  # 1 unit of "Clean coupon"
    "03_Laser welding": load(CT_SCAN_NAME, "5a81aade45924c56854be233364a0435"),  # 1 unit of "Weld coupon"
    "04_Cutting":       load(CT_SCAN_NAME, "7faf722864b94ed797cdd406d22120db"),  # 1 unit of "Weld cross-sections"
    "05_CT-scan":       load(CT_SCAN_NAME, "93d9467878cb43ba9798a256e98512f5"),  # 1 unit of "Tested coupon"
}

# Check the comments against the stored reference flows
reference_flows = pd.DataFrame(
    [{"Database": a["database"], "Process": name, "Reference product": a.get("reference product"), "Location": a.get("location"),
      "Amount": next(e["amount"] for e in a.exchanges() if e["type"] == "production"), "Unit": a["unit"]}
     for procs in (l4m_processes, metallurgic_processes, ct_scan_processes) for name, a in procs.items()])
reference_flows

,Database,Process,Reference product,Location,Amount,Unit
0,L4M system,001_Physical lab (battery cooling pre-processi...,1 set of optimised parameters,GLO,1.0,unit
1,L4M system,001_Physical_Cutting,Cut coupon,GLO,500.0,unit
2,L4M system,001_Physical_Laser welding,Weld coupon,GLO,250.0,unit
3,L4M system,001_Pyshical_Cleaning,Clean coupon,GLO,500.0,unit
4,L4M system,Battery cooling welding,Battery cooling plate,GLO,1.0,unit
5,L4M system,CT-scan X-ray,Tested coupon,GB,1.0,unit
6,L4M system,Digital lab processes,Digital lab processes,GLO,1.0,unit
7,L4M system,Fusion welding,Pipe welding,GLO,1.0,unit
8,L4M system,Pharma welding,Freeze dryer shelf,GLO,1.0,unit
9,L4M system,Platform,Platform,GLO,1.0,unit


In [61]:
USE_CASES = {                       # use case -> its welding process in L4M system
    "Fusion welding":          l4m_processes["Fusion welding"],
    "Pharma welding":          l4m_processes["Pharma welding"],
    "Battery cooling welding": l4m_processes["Battery cooling welding"],
}
digital_lab    = l4m_processes["Digital lab processes"]
coupon_joining = l4m_processes["001_Physical_Laser welding"]    # 250 weld coupons, pulls in the cleaned and cut coupons

# Dynamic Beam Shaping consumes 20% less energy than the standard laser (applied to all electricity of the joining steps)
DBS_ENERGY_FACTOR = 0.8
# PLACEHOLDER: electricity of one optimization run (kWh)
OPTIMIZATION_KWH = 1.0

# PLACEHOLDERS (read from the process diagram): per round, the number of coupon welds, prototype welds, digital lab runs,
# optimization runs and verifications (cutting, polishing, etching and CT-scan of the joined welds).
# "joining" is the technology used for the coupon and prototype joining.
def _round(coupon=0, prototype=0, digital=0, optimization=0, verification=1):
    return dict(coupon=coupon, prototype=prototype, digital=digital, optimization=optimization, verification=verification)

def scenarios(prototype_soa, prototype_l4m):
    return {
        "SoA R&D": {"joining": "Standard laser", "rounds": {
            "Coupon round 1":    _round(coupon=150),
            "Coupon round 2":    _round(coupon=150),
            "Prototype round 1": _round(prototype=prototype_soa)}},
        "L4M-enabled R&D": {"joining": "Dynamic Beam Shaping", "rounds": {
            "Coupon round 1":    _round(coupon=100, digital=1),
            "Coupon round 2":    _round(coupon=100, digital=1, optimization=1),
            "Prototype round 1": _round(prototype=prototype_l4m)}},
    }

RD_SCENARIOS = {uc: scenarios(prototype_soa=50, prototype_l4m=75) for uc in USE_CASES}

# One product system per use case and scenario; reference flow = 1 set of optimised parameters
RD_NAME = "L4M_RD_parameter_sets"
rd_data = {}

def raw_material_sourcing(market):
    """Raw material sourcing: supplies 1 unit of an ecoinvent material, so the process that uses it is a pure service."""
    key = (RD_NAME, "sourcing_" + market["code"])
    if key not in rd_data:
        rd_data[key] = {"name": "Raw material sourcing", "reference product": market.get("reference product") or market["name"],
                        "unit": market["unit"], "location": market.get("location", "GLO"), "technology": "Raw material sourcing",
                        "exchanges": [{"input": key, "amount": 1, "type": "production"},
                                      {"input": market.key, "amount": 1, "type": "technosphere"}]}
    return key

def copy_process(proc, suffix, technology, electricity=1.0, replace=None, drop=(), extra=(), sourcing=False):
    """Copy of a process into the R&D database (name unchanged, code suffixed). `electricity` scales its electricity inputs;
    `replace` maps input keys to other keys, `drop` removes inputs, `extra` adds (key, amount) inputs.
    With `sourcing`, every aluminium input of the process is moved to a Raw material sourcing process."""
    key = (RD_NAME, proc["code"] + suffix)
    exchanges = [{"input": key, "amount": next(e["amount"] for e in proc.exchanges() if e["type"] == "production"), "type": "production"}]
    for e in proc.exchanges():
        if e["type"] == "production" or e["input"] in drop:
            continue
        amount = e["amount"] * (electricity if "electricity" in e.input["name"] else 1)
        target = (replace or {}).get(e["input"], e["input"])
        if sourcing and e["type"] == "technosphere" and e.input["database"] == ECOINVENT_DATABASE_NAME and "aluminium" in e.input["name"]:
            target = raw_material_sourcing(e.input)
        exchanges.append({"input": target, "amount": amount, "type": e["type"]})
    exchanges += [{"input": k, "amount": a, "type": "technosphere"} for k, a in extra]
    rd_data[key] = {"name": proc["name"], "reference product": proc.get("reference product"), "unit": proc["unit"],
                    "location": proc["location"], "technology": technology, "exchanges": exchanges}
    return key

# Coupon preparation: cutting and cleaning are services; the aluminium comes from Raw material sourcing
met_cut, met_clean = metallurgic_processes["01_Cutting"], metallurgic_processes["02_Cleaning"]
cut_service   = copy_process(met_cut, "_svc", "Cutting service", sourcing=True)
clean_service = copy_process(met_clean, "_svc", "Cleaning service", replace={met_cut.key: cut_service})
std_coupon = copy_process(coupon_joining, "_std", "Standard laser", replace={met_clean.key: clean_service})

# DBS versions of the coupon joining and of each use-case welding process
dbs_coupon  = copy_process(coupon_joining, "_DBS", "Dynamic Beam Shaping", DBS_ENERGY_FACTOR, replace={met_clean.key: clean_service})
dbs_welding = {uc: copy_process(p, "_DBS", "Dynamic Beam Shaping", DBS_ENERGY_FACTOR) for uc, p in USE_CASES.items()}

# Verification of the joined welds (no extra weld or cleaning): cutting -> polishing -> etching -> CT-scan
ct_cut   = ct_scan_processes["04_Cutting"]
ct_weld  = next(e.input for e in ct_cut.exchanges() if e["type"] == "technosphere" and e.input["database"] == CT_SCAN_NAME)
polishing, etching = metallurgic_processes["07_Polishing"], metallurgic_processes["08_Etching"]
ver_cut   = copy_process(ct_cut, "_ver", "Verification", drop={ct_weld.key}, sourcing=True)
ver_polish = copy_process(polishing, "_ver", "Verification", extra=[(ver_cut, 6)])
ver_etch  = copy_process(etching, "_ver", "Verification", replace={polishing.key: ver_polish})
ver_ct    = copy_process(l4m_processes["CT-scan X-ray"], "_ver", "Verification", replace={ct_cut.key: ver_etch})

# Optimization of the welding parameters: electricity only (placeholder)
electricity_rer = next(e.input for e in l4m_processes["Platform"].exchanges() if e["type"] == "technosphere")
optimization = (RD_NAME, "optimization")
rd_data[optimization] = {"name": "Optimization", "reference product": "optimization of the welding parameters", "unit": "unit", "location": "GLO",
                         "exchanges": [{"input": optimization, "amount": 1, "type": "production"},
                                       {"input": electricity_rer.key, "amount": OPTIMIZATION_KWH, "type": "technosphere"}]}

for uc, scs in RD_SCENARIOS.items():
    for sc, spec in scs.items():
        dbs = spec["joining"] == "Dynamic Beam Shaping"
        round_keys = []
        for rname, r in spec["rounds"].items():
            rkey = (RD_NAME, f"{uc} | {sc} | {rname}")
            exchanges = [{"input": rkey, "amount": 1, "type": "production"}]
            for count, k in ((r["coupon"], dbs_coupon if dbs else std_coupon),
                             (r["prototype"], dbs_welding[uc] if dbs else USE_CASES[uc].key),
                             (r["digital"], digital_lab.key), (r["optimization"], optimization), (r["verification"], ver_ct)):
                if count:
                    exchanges.append({"input": k, "amount": count, "type": "technosphere"})
            rd_data[rkey] = {"name": rname, "reference product": "round", "unit": "unit", "location": "GLO", "exchanges": exchanges}
            round_keys.append(rkey)
        key = (RD_NAME, f"{uc} | {sc}")
        rd_data[key] = {"name": f"{uc} | {sc}", "reference product": "1 set of optimised parameters", "unit": "set of parameters", "location": "GLO",
                        "exchanges": [{"input": key, "amount": 1, "type": "production"}] + [{"input": k, "amount": 1, "type": "technosphere"} for k in round_keys]}
rd = bd.Database(RD_NAME)
rd.write(rd_data)

systems = {(uc, sc): rd.get(f"{uc} | {sc}") for uc in RD_SCENARIOS for sc in RD_SCENARIOS[uc]}

100%|██████████| 38/38 [00:00<00:00, 9183.20it/s]

14:19:37+0100 [info     ] Vacuuming database            


### Impact calculation
Each product system is walked down through the foreground processes to the background inputs (ecoinvent markets, biosphere flows). Every background input is characterised once per indicator; the walk scales it by the amount needed for 1 set of parameters. The total is checked against a regular `bw2calc` LCA, and the same walk feeds the stacked bar.

In [62]:
M = len(ef31_methods)

def _unit_impacts(act):
    lca = LCA({act: 1}, method=ef31_methods[0]); lca.lci()
    out = []
    for m in ef31_methods:
        lca.switch_method(m); lca.lcia(); out.append(lca.score)
    return np.array(out)

_cf = [{row[0]: row[1] for row in bd.Method(m).load()} for m in ef31_methods]
_q = {}
def background_impact(key, kind):
    if (key, kind) not in _q:
        _q[key, kind] = np.array([cf.get(key, 0.0) for cf in _cf]) if kind == "biosphere" else _unit_impacts(bd.get_activity(key))
    return _q[key, kind]

is_electricity = lambda key: bool(re.match(r"market (group )?for electricity, medium voltage", bd.get_activity(key)["name"]))

def trace(act, mult=1.0, path=()):
    """Walk the foreground below `act` (demanded `mult` times); yield (path of processes, background key, kind, amount)."""
    path = path + (act.key,)
    prod = next(e["amount"] for e in act.exchanges() if e["type"] == "production")
    for e in act.exchanges():
        if e["type"] == "production":
            continue
        amount = mult * e["amount"] / prod
        if e["input"][0] in [*FOREGROUND_DBS, RD_NAME]:
            yield from trace(bd.get_activity(e["input"]), amount, path)
        else:
            yield path, e["input"], e["type"], amount

records = {k: list(trace(act)) for k, act in systems.items()}

# In a DBS scenario no standard-laser weld may remain anywhere in the walk
for (uc, sc), recs in records.items():
    if RD_SCENARIOS[uc][sc]["joining"] == "Dynamic Beam Shaping":
        welds = {u["name"] for u in USE_CASES.values()}
        std = {bd.get_activity(p)["name"] for path, *_ in recs for p in path
               if ("Laser welding" in bd.get_activity(p)["name"] or bd.get_activity(p)["name"] in welds)
               and bd.get_activity(p).get("technology") != "Dynamic Beam Shaping"}
        assert not std, f"Standard-laser weld left in {uc} | {sc}: {std}"

def contributions(k, country=None):
    """One row per background flow: path, key, kind, impact vector. `country` swaps all electricity for that market (needs `grid`)."""
    rows = []
    for path, key, kind, amount in records[k]:
        q = grid[country] if country and kind == "technosphere" and is_electricity(key) else background_impact(key, kind)
        rows.append((path, key, kind, amount * q))
    return rows

# Totals per product system, checked against a regular bw2calc LCA
totals = {k: sum(c[3] for c in contributions(k)) for k in systems}
for k, act in systems.items():
    assert np.allclose(totals[k], _unit_impacts(act), rtol=1e-6), f"Walk does not reproduce bw2calc for {k}"
print(f"Walk matches bw2calc for all {len(systems)} product systems.")

/opt/anaconda3/envs/brightway/lib/python3.11/site-packages/scikits/umfpack/umfpack.py:737: UmfpackWarning: (almost) singular matrix! (estimated cond. number: 7.65e+12)
  warnings.warn(msg, UmfpackWarning)


Walk matches bw2calc for all 6 product systems.


## Stacked bar: SoA R&D vs L4M-enabled R&D
Contribution analysis of the product systems. Impact of 1 set of optimised parameters, split by the process that causes it. Processes are grouped into pre-processing, joining, post-processing, testing and digital lab; the legend follows the same grouping.

In [63]:
# Process types and their processes (legend order); one colour family per type
PROCESS_TYPES = {
    "Pre-processing":  {"Raw material sourcing": "#0f3f7a", "Cutting": "#2a78d6", "Cleaning": "#8fb8ee"},
    "Joining":         {"Coupon joining": "#eb6834", "Prototype joining": "#f4a582"},
    "Post-processing": {"Cutting (weld cross-sections)": "#4b3bb5", "Polishing": "#8d80e0", "Etching": "#b0a6ec"},
    "Testing":         {"CT-scan X-ray": "#1baf7a"},
    "Digital lab":     {"Platform": "#eda100", "Prediction": "#e87ba4", "Optimization": "#f6cf6b"},
}
use_case_welding_names = {p["name"] for p in USE_CASES.values()}

def process_label(key):
    name = bd.get_activity(key)["name"]
    if name in use_case_welding_names:
        return "Prototype joining"
    if name.startswith("04_Cutting"):
        return "Cutting (weld cross-sections)"
    label = re.sub(r"^\d+_(Physical_|Pyshical_)?", "", name)
    return "Coupon joining" if label == "Laser welding" else label

def stack_table(ind):
    """Impact per set of parameters caused directly by each process (rows: systems, columns: process labels)."""
    k = indicators.index(ind)
    out = {}
    for sysk in systems:
        s = {}
        for path, key, kind, v in contributions(sysk):
            lab = process_label(path[-1])
            s[lab] = s.get(lab, 0.0) + v[k]
        out[sysk] = s
    return pd.DataFrame(out).T.fillna(0.0)

w_ind = widgets.Dropdown(options=indicators, value=DEFAULT_INDICATOR, description="Indicator", layout=widgets.Layout(width="520px"), style={"description_width": "80px"})
fig_bar = go.FigureWidget()

def draw_bars(_=None):
    d = stack_table(w_ind.value)
    unknown = [c for c in d.columns if not any(c in procs for procs in PROCESS_TYPES.values())]
    assert not unknown, f"Processes without a type: {unknown}"
    unit = method_units.get(w_ind.value, "")
    x = [[uc for uc, _ in d.index], [sc for _, sc in d.index]]
    tot = d.sum(axis=1)
    with fig_bar.batch_update():
        fig_bar.data = []
        for ptype, procs in PROCESS_TYPES.items():
            first = True
            for proc, color in procs.items():
                if proc not in d.columns:
                    continue
                fig_bar.add_bar(x=x, y=d[proc], name=proc, legendgroup=ptype, legendgrouptitle_text=f"<b>{ptype}</b>" if first else None,
                                marker=dict(color=color, line=dict(color="white", width=1)),
                                customdata=(d[proc] / tot * 100),
                                hovertemplate=f"<b>%{{x}}</b><br>{ptype}: {proc}<br>%{{y:.3g}} {unit} (%{{customdata:.1f}}% of total)<extra></extra>")
                first = False
        fig_bar.add_scatter(x=x, y=tot, mode="text", text=[f"{t:.3g}" for t in tot], textposition="top center", showlegend=False, hoverinfo="skip")
        total_trace = fig_bar.data[-1]
        bar_traces = list(fig_bar.data[:-1])

        def update_totals(*_):
            # Total labels follow the visible legend items, so the y axis rescales when items are deselected
            vis = [t for t in bar_traces if t.visible in (True, None)]
            new_tot = sum((np.asarray(t.y, dtype=float) for t in vis), np.zeros(len(d)))
            with fig_bar.batch_update():
                total_trace.y = new_tot
                total_trace.text = [f"{v:.3g}" for v in new_tot]

        for t in bar_traces:
            t.on_change(update_totals, "visible")
        fig_bar.update_layout(barmode="relative", template="plotly_white", height=900, width=1260, font=dict(size=14),
                              title=dict(text=f"{short(w_ind.value)} per set of optimised parameters", x=0.01, font=dict(size=15)),
                              yaxis=dict(title=unit, gridcolor="#e6e9ec"), legend=dict(groupclick="toggleitem", tracegroupgap=14),
                              margin=dict(l=70, r=20, t=80, b=50))

w_ind.observe(draw_bars, names="value")
draw_bars()
display(widgets.VBox([w_ind, fig_bar]))

## Sankey: where the impact flows
Background inputs (left) feed the foreground processes, which pass their cumulative impact on to the set of parameters (right).

## Regional variation: EU electricity markets
Every electricity input (the RER market group and the GB markets used by the physical lab) is replaced by one country's market. Left: SoA R&D, right: L4M-enabled R&D.